In [0]:
from pyspark.sql import Row
from datetime import datetime

In [0]:
%sql
create or replace table aws_databricks.raw.customers(
    customer_id int,
    name string,
    email string,
    tier string,
    updated_at timestamp
)
using delta;

insert into aws_databricks.raw.customers values (1,"John","john@gmail.com","bronze","2022-01-01"),
(2,"Jane","jane@gmail.com","silver","2022-01-02"),
(3,"Jack","jack@gmail.com","gold","2022-01-03"),
(4,"Jill","jill@gmail.com","platinum","2022-01-04"),
(5,"Joe","joe@gmail.com","bronze","2022-01-05");

In [0]:
source_df = [
    Row(
        customer_id = 1,
        name  = "John",
        email = "john@example.com",
        tier =  "silver",
        updated_at = datetime(2025,1,1,12,00)
),Row(
        customer_id = 3,
        name  = "Jack",
        email = "jack@example.com",
        tier =  "platinum",
        updated_at = datetime(2025,1,1,12,00)
),
Row(
    customer_id = 6,
    name = "rama",
    email = "rama@gmail.com",
    tier = "gold",
    updated_at = datetime(2025,12,1,14,1,00)
)
]

In [0]:
df = spark.createDataFrame(source_df)

In [0]:
df.createOrReplaceTempView("cust_update")

In [0]:
%sql
merge into aws_databricks.raw.customers t
using cust_update s
on t.customer_id = s.customer_id
when matched then update set
    t.tier = s.tier,
    t.updated_at = s.updated_at
when not matched then insert (customer_id,name,email,tier,updated_at) values (s.customer_id,s.name,s.email,s.tier,s.updated_at)

In [0]:
%sql
create or replace table aws_databricks.raw.crm(
    cust_id int,
    name string,
    email string,
    tier string,
    updated_at timestamp
);

insert into aws_databricks.raw.crm values (1,"John","john@gmail.com","bronze","2025-01-01 12:00:00"),
(2,"Jane","jane@gmail.com","silver","2025-01-01 12:00:00"),
(3,"Jack","jack@gmail.com","gold","2025-01-01 12:00:00"),
(4,"Jill","jill@gmail.com","platinum","2025-01-01 12:00:00"),
(5,"Joe","joe@gmail.com","bronze","2025-01-01 12:00:00");

In [0]:
%sql
select * from aws_databricks.raw.crm;

In [0]:
crm_source = ([
    Row(
        customer_id = 2,
        name = 'Jane',
        email = 'jane@gmail.com',
        tier = "bronze",
        updated_at = datetime(2026,1,1,12,0,0)
),
    Row(
        customer_id = 4,
        name = 'Jill',
        email = 'jill@gmail.com',
        tier = "gold",
        updated_at = datetime(2024,12,31,12,0,0)
),
    Row(
        customer_id = 6,
        name = 'ram',
        email = 'ram@gmail.com',
        tier = "gold",
        updated_at = datetime(2026,1,1,12,0,0)
)
])

In [0]:
df = spark.createDataFrame(crm_source)
df.createOrReplaceTempView("crm_source")

In [0]:
%sql
select * from crm_source

In [0]:
%sql
merge into aws_databricks.raw.crm t
using crm_source s
on t.cust_id = s.customer_id
when matched and t.updated_at < s.updated_at then
update set t.tier = s.tier,
           t.updated_at = s.updated_at
when not matched then 
insert (t.cust_id,t.name,t.email,t.tier,t.updated_at) 
values (s.customer_id,s.name,s.email,s.tier,s.updated_at)

In [0]:
%sql
select * from aws_databricks.raw.crm